# 05b — Feature Validation (Phase 3 follow-up)

This notebook answers a specific question the specification implies but does
not spell out: **are the engineered features meaningful?**

We compute three diagnostics on the training split only:

1. **Target signal** — mutual information with the 3-class target (classification)
   or |Pearson| correlation with `int_rate` (regression).
2. **Redundancy** — max |Spearman| correlation between each engineered feature
   and any original feature.
3. **Self-collinearity** — max |Spearman| correlation between each feature and
   any other feature in the set.

Outputs feed a validation subsection of the mid-journey report and directly
inform which engineered features to keep in Phase 4.

No model is trained here.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_validation import FeatureValidator
from src.splits import ChronologicalSplitter

sns.set_theme(style='whitegrid')
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'; OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIG    = PROJECT_ROOT / 'outputs' / 'figures'; OUT_FIG.mkdir(parents=True, exist_ok=True)
SEED = 42
validator = FeatureValidator(seed=SEED)
print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados


## 1. Load engineered cohorts and reuse the frozen split

In [2]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)
split_cfg

{'date_col': 'issue_d', 'train_end': '2014-12', 'val_end': '2016-12'}

In [3]:
splitter = ChronologicalSplitter(**split_cfg)

acc_clf_fe = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification_fe.parquet')
acc_reg_fe = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression_fe.parquet')

splits_clf = splitter.split(acc_clf_fe)
splits_reg = splitter.split(acc_reg_fe)

train_clf = splits_clf.train.copy()
train_reg = splits_reg.train.copy()

print('train_clf:', train_clf.shape)
print('train_reg:', train_reg.shape)

train_clf: (451671, 167)
train_reg: (466345, 164)


## 2. Define engineered and original feature lists

In [4]:
ENGINEERED = [
    'loan_to_income', 'dti_x_loan_to_income', 'credit_hist_months',
    'revol_util_bucket', 'total_credit_lines', 'inquiries_per_open_account',
    'util_x_fico', 'bc_pressure', 'delinq_rate', 'loan_per_term',
    'emp_length_years', 'has_bankruptcy', 'closed_credit_share',
    'installment_burden', 'rate_burden',
]
ENGINEERED = [c for c in ENGINEERED if c in train_clf.columns]

ORIGINALS = [
    'loan_amnt', 'term', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'pub_rec',
    'pub_rec_bankruptcies', 'mort_acc', 'revol_bal', 'revol_util',
    'tot_cur_bal', 'tot_hi_cred_lim', 'bc_util', 'bc_open_to_buy',
    'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'pct_tl_nvr_dlq',
    'installment', 'int_rate',
]
ORIGINALS = [c for c in ORIGINALS if c in train_clf.columns]

print('engineered:', len(ENGINEERED))
print('originals :', len(ORIGINALS))

engineered: 15
originals : 24


## 3. Spearman correlation heatmap: engineered vs original

Any engineered feature with a high |correlation| against a single original is
near-redundant and unlikely to add new information for linear models.

In [5]:
corr_block = train_clf[ENGINEERED + ORIGINALS].corr(method='spearman', numeric_only=True)
cross = corr_block.loc[ENGINEERED, ORIGINALS]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(cross, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=False, cbar_kws={'label': 'Spearman r'}, ax=ax)
ax.set_title('Engineered (rows) vs original (columns) — Spearman correlation')
ax.set_xlabel('Original features')
ax.set_ylabel('Engineered features')
fig.tight_layout()
fig.savefig(OUT_FIG / 'fe_vs_original_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.close(fig)
cross.round(3).to_csv(OUT_TABLES / 'fe_vs_original_corr.csv')
print('saved fe_vs_original_corr_heatmap.png and fe_vs_original_corr.csv')
cross.round(2)

saved fe_vs_original_corr_heatmap.png and fe_vs_original_corr.csv


,loan_amnt,term,annual_inc,dti,fico_range_low,fico_range_high,delinq_2yrs,inq_last_6mths,open_acc,total_acc,...,revol_util,tot_cur_bal,tot_hi_cred_lim,bc_util,bc_open_to_buy,mo_sin_old_rev_tl_op,num_actv_rev_tl,pct_tl_nvr_dlq,installment,int_rate
loan_to_income,0.62,0.33,-0.32,0.26,0.03,0.03,-0.06,-0.08,0.02,-0.03,...,0.07,-0.10,-0.10,0.05,0.06,0.00,0.09,0.13,0.59,0.21
dti_x_loan_to_income,0.46,0.27,-0.31,0.74,-0.02,-0.02,-0.04,-0.05,0.21,0.12,...,0.16,-0.00,-0.00,0.13,0.00,0.02,0.21,0.12,0.45,0.22
credit_hist_months,0.21,0.08,0.27,0.05,0.09,0.09,0.12,-0.00,0.17,0.36,...,0.01,0.22,0.25,-0.02,0.15,0.90,0.14,-0.17,0.19,-0.08
revol_util_bucket,0.11,0.08,0.07,0.17,-0.37,-0.37,-0.02,-0.09,-0.13,-0.10,...,0.95,0.09,0.01,0.77,-0.56,-0.01,0.07,0.01,0.13,0.31
total_credit_lines,0.26,0.11,0.34,0.28,0.02,0.02,0.13,0.13,0.82,0.97,...,-0.12,0.41,0.45,-0.11,0.25,0.31,0.45,-0.07,0.24,-0.03
inquiries_per_open_account,-0.06,0.01,0.02,-0.05,-0.08,-0.08,0.01,0.96,-0.07,0.01,...,-0.06,0.00,-0.00,-0.06,-0.02,-0.03,-0.05,-0.05,-0.04,0.22
util_x_fico,0.13,0.08,0.08,0.18,-0.32,-0.32,-0.04,-0.10,-0.14,-0.10,...,1.00,0.12,0.03,0.80,-0.56,0.01,0.07,0.04,0.15,0.30
bc_pressure,-0.17,-0.00,-0.17,0.11,-0.47,-0.47,0.02,-0.04,-0.23,-0.18,...,0.67,-0.11,-0.20,0.88,-0.96,-0.15,-0.03,-0.11,-0.14,0.37
delinq_rate,0.01,0.00,0.08,-0.01,-0.22,-0.22,1.00,0.01,0.04,0.11,...,-0.02,0.08,0.09,-0.02,-0.04,0.11,-0.02,-0.50,0.02,0.09
loan_per_term,0.92,0.06,0.48,0.04,0.14,0.14,0.01,-0.03,0.21,0.23,...,0.09,0.32,0.34,0.01,0.25,0.19,0.15,0.07,0.98,-0.04


## 4. Redundancy table

For each engineered feature, the strongest |Spearman| correlation with any
original and whether it crosses the near-redundancy threshold (0.95).

In [6]:
red = validator.redundancy_table(train_clf, ENGINEERED, ORIGINALS, method='spearman', threshold=0.95)
red.to_csv(OUT_TABLES / 'fe_redundancy_with_originals.csv', index=False)
red

,engineered,max_abs_corr_with_original,partner_original,near_redundant
0,has_bankruptcy,0.999765,pub_rec_bankruptcies,True
1,util_x_fico,0.996591,revol_util,True
2,delinq_rate,0.996422,delinq_2yrs,True
3,loan_per_term,0.976262,installment,True
4,total_credit_lines,0.973232,total_acc,True
5,inquiries_per_open_account,0.964879,inq_last_6mths,True
6,bc_pressure,0.959708,bc_open_to_buy,True
7,revol_util_bucket,0.954871,revol_util,True
8,credit_hist_months,0.899584,mo_sin_old_rev_tl_op,False
9,rate_burden,0.895371,loan_amnt,False


## 5. Target-signal diagnostics on classification cohort

Mutual information of each feature (engineered and original) with the 3-class
target. MI is computed on the training split only.

In [7]:
y_clf = train_clf['target_class']
diag_clf = validator.build_diagnostics(
    train_clf, ENGINEERED, ORIGINALS, y_clf,
    task='classification',
    categorical_cols=['home_ownership','verification_status','purpose','addr_state','application_type'],
    seed=SEED,
)
diag_clf.to_csv(OUT_TABLES / 'fe_diagnostics_classification.csv', index=False)
print('saved fe_diagnostics_classification.csv')
diag_clf.round(4)

saved fe_diagnostics_classification.csv


,name,kind,target_signal,max_redundancy,max_redundancy_with,max_self_collinearity,max_self_collinearity_with
0,revol_util_bucket,engineered,0.0271,0.9549,revol_util,0.9549,revol_util
1,rate_burden,engineered,0.0245,0.8954,loan_amnt,0.8954,loan_amnt
2,emp_length_years,engineered,0.0151,0.2476,mo_sin_old_rev_tl_op,0.2739,credit_hist_months
3,installment_burden,engineered,0.0147,0.5756,installment,0.9581,loan_to_income
4,loan_per_term,engineered,0.0139,0.9763,installment,0.9763,installment
5,dti_x_loan_to_income,engineered,0.0104,0.7411,dti,0.8112,loan_to_income
6,loan_to_income,engineered,0.0102,0.6160,loan_amnt,0.9581,installment_burden
7,util_x_fico,engineered,0.0090,0.9966,revol_util,0.9966,revol_util
8,bc_pressure,engineered,0.0033,0.9597,bc_open_to_buy,0.9597,bc_open_to_buy
9,inquiries_per_open_account,engineered,0.0022,0.9649,inq_last_6mths,0.9649,inq_last_6mths


## 6. Target-signal bar chart — engineered vs original

In [8]:
plot_df = diag_clf.copy().sort_values('target_signal', ascending=True)
colors = plot_df['kind'].map({'engineered': '#DD8452', 'original': '#4C72B0'})

fig, ax = plt.subplots(figsize=(9, max(6, 0.32 * len(plot_df))))
ax.barh(plot_df['name'], plot_df['target_signal'], color=colors)
ax.set_xlabel('Mutual information with target_class')
ax.set_title('Target signal — engineered (orange) vs original (blue)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#4C72B0', label='original'),
                   Patch(color='#DD8452', label='engineered')])
fig.tight_layout()
fig.savefig(OUT_FIG / 'fe_target_signal_bar.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('saved fe_target_signal_bar.png')

saved fe_target_signal_bar.png


## 7. Quadrant scatter — redundancy vs target signal

The ideal engineered feature sits in the **top-left**: low redundancy with
originals, high target signal.

In [9]:
eng_df = diag_clf[diag_clf['kind'] == 'engineered'].copy()
eng_df = eng_df.dropna(subset=['max_redundancy', 'target_signal'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(eng_df['max_redundancy'], eng_df['target_signal'],
           s=90, color='#DD8452', edgecolor='black')
for _, row in eng_df.iterrows():
    ax.annotate(row['name'], (row['max_redundancy'], row['target_signal']),
                fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.axvline(0.95, color='red', linestyle='--', linewidth=1,
           label='redundancy threshold (0.95)')
ax.set_xlabel('Max |Spearman| with any original feature')
ax.set_ylabel('Mutual information with target_class')
ax.set_title('Engineered features: redundancy vs target signal')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_FIG / 'fe_quadrant.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('saved fe_quadrant.png')

saved fe_quadrant.png


In [10]:
eng_df['verdict'] = eng_df.apply(validator.verdict, axis=1)
eng_df.to_csv(OUT_TABLES / 'fe_verdicts_classification.csv', index=False)
eng_df[['name','target_signal','max_redundancy','max_redundancy_with','verdict']]

,name,target_signal,max_redundancy,max_redundancy_with,verdict
0,revol_util_bucket,0.027121,0.954871,revol_util,near-redundant — drop or keep only one
1,rate_burden,0.024534,0.895371,loan_amnt,keep — non-redundant with reasonable signal
2,emp_length_years,0.015055,0.247617,mo_sin_old_rev_tl_op,keep — non-redundant with reasonable signal
3,installment_burden,0.014746,0.575565,installment,keep — non-redundant with reasonable signal
4,loan_per_term,0.013909,0.976262,installment,near-redundant — drop or keep only one
5,dti_x_loan_to_income,0.010367,0.741132,dti,keep — non-redundant with reasonable signal
6,loan_to_income,0.010156,0.615957,loan_amnt,keep — non-redundant with reasonable signal
7,util_x_fico,0.008964,0.996591,revol_util,near-redundant — drop or keep only one
8,bc_pressure,0.003290,0.959708,bc_open_to_buy,near-redundant — drop or keep only one
9,inquiries_per_open_account,0.002188,0.964879,inq_last_6mths,near-redundant — drop or keep only one


## 8. Multicollinearity among engineered features

High Spearman correlation signals similar rankings, not necessarily identical
variables. Exact rescalings were removed. Correlated features require care for
linear models; usefulness for trees or interactions still requires validation.

In [11]:
col = validator.self_collinearity(train_clf, ENGINEERED, method='spearman')
col.to_csv(OUT_TABLES / 'fe_self_collinearity.csv', index=False)
col

,feature,max_self_collinearity,partner
0,loan_to_income,0.958139,installment_burden
1,installment_burden,0.958139,loan_to_income
2,revol_util_bucket,0.951603,util_x_fico
3,util_x_fico,0.951603,revol_util_bucket
4,dti_x_loan_to_income,0.811212,loan_to_income
5,loan_per_term,0.754124,rate_burden
6,rate_burden,0.754124,loan_per_term
7,bc_pressure,0.653863,util_x_fico
8,credit_hist_months,0.325962,total_credit_lines
9,total_credit_lines,0.325962,credit_hist_months


## 9. Regression diagnostics

Use only pre-pricing eligible features: exclude funding-date credit history,
installment burden and rate burden. Signal is absolute Pearson correlation with
interest rate; it is not directly comparable with classification mutual information.

In [12]:
ENGINEERED_REG = [c for c in ENGINEERED if c not in {'installment_burden','rate_burden','credit_hist_months'}]
ORIGINALS_REG  = [c for c in ORIGINALS if c not in {'installment','int_rate'}]
y_reg = train_reg['int_rate']

diag_reg = validator.build_diagnostics(
    train_reg, ENGINEERED_REG, ORIGINALS_REG, y_reg,
    task='regression',
    categorical_cols=['home_ownership','verification_status','purpose','addr_state','application_type'],
    seed=SEED,
)
diag_reg.to_csv(OUT_TABLES / 'fe_diagnostics_regression.csv', index=False)
print('saved fe_diagnostics_regression.csv')
diag_reg.round(4)

saved fe_diagnostics_regression.csv


,name,kind,target_signal,max_redundancy,max_redundancy_with,max_self_collinearity,max_self_collinearity_with
0,revol_util_bucket,engineered,0.3061,0.9550,revol_util,0.9550,revol_util
1,util_x_fico,engineered,0.2919,0.9966,revol_util,0.9966,revol_util
2,dti_x_loan_to_income,engineered,0.2360,0.7430,dti,0.8120,loan_to_income
3,loan_to_income,engineered,0.2182,0.6146,loan_amnt,0.8120,dti_x_loan_to_income
4,inquiries_per_open_account,engineered,0.1625,0.9655,inq_last_6mths,0.9655,inq_last_6mths
5,delinq_rate,engineered,0.0838,0.9964,delinq_2yrs,0.9964,delinq_2yrs
6,closed_credit_share,engineered,0.0812,0.4933,total_acc,0.4933,total_acc
7,has_bankruptcy,engineered,0.0585,0.9998,pub_rec_bankruptcies,0.9998,pub_rec_bankruptcies
8,bc_pressure,engineered,0.0567,0.9600,bc_open_to_buy,0.9600,bc_open_to_buy
9,loan_per_term,engineered,0.0424,0.9173,loan_amnt,0.9173,loan_amnt


In [13]:
plot_df = diag_reg.copy().sort_values('target_signal', ascending=True)
colors = plot_df['kind'].map({'engineered': '#DD8452', 'original': '#4C72B0'})

fig, ax = plt.subplots(figsize=(9, max(6, 0.32 * len(plot_df))))
ax.barh(plot_df['name'], plot_df['target_signal'], color=colors)
ax.set_xlabel('|Pearson correlation| with int_rate')
ax.set_title('Target signal (regression) — engineered (orange) vs original (blue)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#4C72B0', label='original'),
                   Patch(color='#DD8452', label='engineered')])
fig.tight_layout()
fig.savefig(OUT_FIG / 'fe_target_signal_bar_regression.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('saved fe_target_signal_bar_regression.png')

saved fe_target_signal_bar_regression.png


## 10. Verdict summary

This prints a compact keep/drop recommendation based on the diagnostics.
The recommendation is descriptive, not prescriptive — tree ensembles can
safely retain near-redundant features.

In [14]:
print('--- Classification cohort verdicts ---')
for _, row in eng_df.sort_values('verdict').iterrows():
    print(f"{row['name']:25s} MI={row['target_signal']:.4f}  "
          f"red={row['max_redundancy']:.3f} with {row['max_redundancy_with']:>20s}  "
          f"-> {row['verdict']}")

print()
print('--- Regression cohort verdicts ---')
eng_reg = diag_reg[diag_reg['kind'] == 'engineered'].copy()
eng_reg = eng_reg.dropna(subset=['max_redundancy', 'target_signal'])
eng_reg['verdict'] = eng_reg.apply(validator.verdict_reg, axis=1)
eng_reg.to_csv(OUT_TABLES / 'fe_verdicts_regression.csv', index=False)
for _, row in eng_reg.sort_values('verdict').iterrows():
    print(f"{row['name']:25s} |r|={row['target_signal']:.4f}  "
          f"red={row['max_redundancy']:.3f} with {row['max_redundancy_with']:>20s}  "
          f"-> {row['verdict']}")

--- Classification cohort verdicts ---
rate_burden               MI=0.0245  red=0.895 with            loan_amnt  -> keep — non-redundant with reasonable signal
emp_length_years          MI=0.0151  red=0.248 with mo_sin_old_rev_tl_op  -> keep — non-redundant with reasonable signal
installment_burden        MI=0.0147  red=0.576 with          installment  -> keep — non-redundant with reasonable signal
dti_x_loan_to_income      MI=0.0104  red=0.741 with                  dti  -> keep — non-redundant with reasonable signal
loan_to_income            MI=0.0102  red=0.616 with            loan_amnt  -> keep — non-redundant with reasonable signal
revol_util_bucket         MI=0.0271  red=0.955 with           revol_util  -> near-redundant — drop or keep only one
loan_per_term             MI=0.0139  red=0.976 with          installment  -> near-redundant — drop or keep only one
util_x_fico               MI=0.0090  red=0.997 with           revol_util  -> near-redundant — drop or keep only one
bc_press

## 11. Feature-validation interpretation

The tables above rank training-only signal and identify potential redundancy.
The printed recommendations are screening heuristics, not empirical proof that a
feature improves a model. Inquiries per open account and closed credit share
combine distinct baseline measures rather than rescale one column. Funding-date
history is classification-only. Keep/drop/interaction counts are printed below;
final predictive selection must be repeated inside training folds without test data.

In [15]:
for task, frame in [('classification', eng_df), ('regression', eng_reg)]:
    print(task, 'screening counts:', frame['verdict'].str.split(' — ').str[0].value_counts().to_dict())
    print('Highest training signal:', frame.sort_values('target_signal', ascending=False)[['name','target_signal']].head(3).to_dict('records'))


classification screening counts: {'near-redundant': 8, 'keep': 5, 'weak signal': 2}
Highest training signal: [{'name': 'revol_util_bucket', 'target_signal': 0.027121425325106863}, {'name': 'rate_burden', 'target_signal': 0.024534382182789694}, {'name': 'emp_length_years', 'target_signal': 0.015054761749902612}]
regression screening counts: {'near-redundant': 7, 'keep': 5}
Highest training signal: [{'name': 'revol_util_bucket', 'target_signal': 0.30608471671000903}, {'name': 'util_x_fico', 'target_signal': 0.291879021166073}, {'name': 'dti_x_loan_to_income', 'target_signal': 0.23597462242963194}]
